# ClinicalBERT

BERT base trained on MIMIC-III clinical notes.

**Model type:** Encoder (masked LM)  
**Pipeline:** Probing → Orthogonalization → Representation steering  
**Note:** Text generation is not available for encoder models. Steering is evaluated by measuring how injecting the probe direction shifts hidden-state projections onto the uncertainty axis.  
**Results saved to:** `results/clinicalbert/`

## 1. Setup

In [ ]:
import sys, json
from pathlib import Path

# ── ensure project root is on the path ──────────────────────────────────────
ROOT = Path().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from src.model_registry  import get_config, list_models
from src.model_loader    import load_model, get_device
from src.bioscope_parser import build_balanced_contrast_set
from src.probing         import run_probing_sweep, best_layer
from src.orthogonalization import build_orthogonal_directions
from src.steering_core   import calibrate_hidden_norms
from src.experiment_runner import run_experiment, PROMPTS, ALPHAS, SEEDS
from src.results_manager import save_results, load_results, results_exist
from src.metrics         import load_english_vocab

print(f"PyTorch   : {torch.__version__}")
print(f"Device    : {get_device()}")


In [ ]:
MODEL_NAME   = "clinicalbert"
DISPLAY_NAME = "ClinicalBERT"
cfg          = get_config(MODEL_NAME)
DEVICE       = get_device()
print(cfg)

## 2. Load model

In [ ]:
tok, model = load_model(MODEL_NAME, device=DEVICE)
print(f"Loaded: {cfg['hf_id']}")

## 3. Load BioScope data

In [ ]:
# ── BioScope contrast set ────────────────────────────────────────────────────
# seed=42 is fixed from the original paper — do not change
uncertain, certain = build_balanced_contrast_set(
    "data/bioscope", max_per_class=200, seed=42
)
print(f"Loaded {len(uncertain)} uncertain + {len(certain)} certain sentences")
print("\nUncertain sample:")
for s in uncertain[:2]:
    print(f"  {s[:100]}")
print("\nCertain sample:")
for s in certain[:2]:
    print(f"  {s[:100]}")


## 4. Probing sweep — all layers

In [ ]:
probe_results = run_probing_sweep(
    uncertain, certain, tok, model,
    layers=cfg["probe_layers"], device=DEVICE,
)

### Best layer

In [ ]:
selected_layer = best_layer(probe_results)
print(f"Selected layer: {selected_layer}  "  
      f"acc={probe_results[selected_layer]['mean_acc']:.2%}")

## 5. Orthogonalization

In [ ]:
ortho = build_orthogonal_directions(
    uncertain, certain, tok, model, selected_layer,
    device=DEVICE,
)
print(f"  cos(probe, length) = {ortho['cos_probe_length']:.4f}")
print(f"  cos(probe, hedge)  = {ortho['cos_probe_hedge']:.4f}")
print(f"  ortho_LH acc       = {ortho['ortho_LH_classification_acc']:.2%}")

## 6. Calibrate hidden norms

In [ ]:
hidden_norms = calibrate_hidden_norms(
    tok, model, MODEL_NAME, layers=[selected_layer], device=DEVICE,
)
print(f"Hidden norm at layer {selected_layer}: "  
      f"{hidden_norms[selected_layer]:.1f}")

## 7. Representation-steering experiment

For each α, inject the steering vector at the selected layer and measure
the mean shift in probe-direction projection across 80 balanced test sentences.

A positive Δ means the representation moves toward the 'uncertain' pole.

In [ ]:
if results_exist(MODEL_NAME, "experiment"):
    print("Loading cached results...")
    results = load_results(MODEL_NAME, "experiment")
else:
    results = run_experiment(
        MODEL_NAME, tok, model, uncertain, certain,
        device=DEVICE,
    )
    save_results(MODEL_NAME, "experiment", results)

## 8. Visualizations

In [ ]:
# ── probe accuracy plot ───────────────────────────────────────────────────────
probe_res = results["probe_results"]
layers_sorted = sorted(int(k) for k in probe_res.keys())
means  = [probe_res[str(l)]["mean_acc"] for l in layers_sorted]
cis_lo = [probe_res[str(l)]["ci_low"]   for l in layers_sorted]
cis_hi = [probe_res[str(l)]["ci_high"]  for l in layers_sorted]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(layers_sorted, means, marker="o", linewidth=2, label="CV accuracy")
ax.fill_between(layers_sorted, cis_lo, cis_hi, alpha=0.2, label="95% CI")
ax.axhline(0.5, color="grey", linestyle="--", linewidth=1, label="Chance")
ax.set_xlabel("Layer")
ax.set_ylabel("Accuracy")
ax.set_title(f"Probe accuracy per layer — {DISPLAY_NAME}")
ax.legend()
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
plt.tight_layout()

fig_path = Path("results") / MODEL_NAME / "fig_probe_accuracy.png"
fig.savefig(fig_path, dpi=150, bbox_inches="tight")
fig.savefig(fig_path.with_suffix(".pdf"), bbox_inches="tight")
plt.show()
print(f"Saved → {fig_path}")


In [ ]:
# ── representation steering plot (encoder) ───────────────────────────────────
summary = results["summary"]
directions = list(summary.keys())

fig, ax = plt.subplots(figsize=(7, 4))
colors  = {"probe": "#2196F3", "ortho": "#FF5722"}
markers = {"probe": "o",       "ortho": "s"}

for dir_name in directions:
    dir_data = summary[dir_name]
    alphas_sorted = sorted(float(a) for a in dir_data.keys())
    deltas = [dir_data[str(a)]["delta_mean"] for a in alphas_sorted]
    stds   = [dir_data[str(a)]["delta_std"]  for a in alphas_sorted]
    ax.plot(alphas_sorted, deltas,
            marker=markers[dir_name], color=colors[dir_name],
            label=dir_name, linewidth=2)
    ax.fill_between(
        alphas_sorted,
        [d - s for d, s in zip(deltas, stds)],
        [d + s for d, s in zip(deltas, stds)],
        alpha=0.15, color=colors[dir_name],
    )

ax.axhline(0, color="grey", linestyle="--", linewidth=1)
ax.set_xlabel("α fraction")
ax.set_ylabel("Mean Δ probe projection")
ax.set_title(f"Representation steering — {DISPLAY_NAME}")
ax.legend()
plt.tight_layout()

fig_path = Path("results") / MODEL_NAME / "fig_representation_steering.png"
fig.savefig(fig_path, dpi=150, bbox_inches="tight")
fig.savefig(fig_path.with_suffix(".pdf"), bbox_inches="tight")
plt.show()
print(f"Saved → {fig_path}")


## 9. Summary statistics

In [ ]:
summary = results["summary"]
header = "{:>8} {:>14} {:>14}".format("", "probe Dproj", "ortho Dproj")
print(header)
for af in [0.0, 0.05, 0.10, 0.15, 0.20, 0.25]:
    pd_ = summary.get("probe", {}).get(str(af), {}).get("delta_mean", float("nan"))
    od_ = summary.get("ortho", {}).get(str(af), {}).get("delta_mean", float("nan"))
    print(f"  alpha={af:.3f}  {pd_:>14.4f} {od_:>14.4f}")


## 10. CLS-token projection distribution

Visualise where uncertain vs. certain sentences fall on the probe axis before and after steering.

In [ ]:
import numpy as np
from src.probing import extract_features

v_probe = np.array(ortho["v_probe"])
v_unit  = v_probe / np.linalg.norm(v_probe)

X_u = extract_features(uncertain[:50], tok, model, selected_layer, DEVICE)
X_c = extract_features(certain[:50],   tok, model, selected_layer, DEVICE)

proj_u = X_u @ v_unit
proj_c = X_c @ v_unit

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(proj_u, bins=20, alpha=0.6, label="Uncertain", color="#E53935")
ax.hist(proj_c, bins=20, alpha=0.6, label="Certain",   color="#1E88E5")
ax.set_xlabel("Probe projection")
ax.set_title(f"Uncertainty axis projection — {DISPLAY_NAME}")
ax.legend()
plt.tight_layout()
fig_path = Path("results") / MODEL_NAME / "fig_probe_projection.png"
fig.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.show()